In [ ]:
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import scvi
import torch
import warnings
import leidenalg as la
import anndata
from matplotlib import pyplot as plt
from matplotlib.pyplot import rc_context
from scvi.autotune import ModelTuner
from ray import tune
import glob
import re
import csv
import itertools
from scipy.io import mmwrite


def _resolve_partition_root():
    cwd = Path(os.getcwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "primary_script").exists() and (p / "intermediate").exists():
            return p
        candidate = p / "technical_review" / "submission_partition_draft"
        if (candidate / "primary_script").exists():
            return candidate
    raise RuntimeError("Could not locate submission_partition_draft root")


REPO_ROOT = _resolve_partition_root()
PBMC_INTERMEDIATE = REPO_ROOT / "intermediate" / "pbmc"
PBMC_INTERMEDIATE.mkdir(parents=True, exist_ok=True)


In [ ]:
print('scanpy version:', sc.__version__)

In [ ]:
import pynndescent
print('pynndescent version:', pynndescent.__version__)

In [ ]:
import umap
print('umap version:', umap.__version__)

In [ ]:
warnings.filterwarnings('ignore')
sc.set_figure_params(dpi=200)
plt.rcParams['figure.figsize'] = [3,3]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
os.chdir(str(REPO_ROOT / 'intermediate'))


In [ ]:
myeloid_platelet = sc.read_h5ad(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_dbl.h5ad'))
myeloid_platelet.obs['cell_group'].value_counts()


### Myeloid

In [ ]:
assert np.all(myeloid_platelet.layers['counts'].data == myeloid_platelet.layers['counts'].data.astype(np.int32))
# start fresh
adata = anndata.AnnData(X = myeloid_platelet.layers['counts'], obs = myeloid_platelet.obs, var = myeloid_platelet.raw.var)
sc.pp.filter_cells(adata, min_genes = 200)
sc.pp.filter_genes(adata, min_cells = 10)
adata.var['MT'] = adata.var_names.str.startswith('MT-')
adata = adata[adata.obs.pct_counts_MT <= 10]
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=4000, subset = False, layer = 'counts', 
                            flavor = "seurat_v3", batch_key="lane")
adata_var = adata.var
csv_data = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'EXCLUDE_XY_TCR_IG.csv'), header=None)

positions = np.where(adata.var_names.isin(csv_data[0]))[0]
adata.var.iloc[positions, adata.var.columns.get_loc('highly_variable')] = False
adata.var['highly_variable'].value_counts()


In [ ]:
model_cls = scvi.model.SCVI
model_cls.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                        categorical_covariate_keys=['study_id'], 
                        continuous_covariate_keys=['pct_counts_MT', 'total_counts'])
tuner = ModelTuner(model_cls)
search_space = {
    "n_hidden": tune.choice([92, 128, 192, 256]),
    "n_latent": tune.choice([10, 20, 30, 40, 50, 60]),
    "n_layers": tune.choice([1, 2, 3]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "gene_likelihood": tune.choice(["nb", "zinb"])}
results = tuner.fit(adata, metric="validation_loss",
                    resources = {'gpu': 1}, 
                    search_space = search_space,
                    num_samples = 100,
                    max_epochs = 20)

In [ ]:
import ray
ray.shutdown()

In [ ]:
import math
import re

def extract_from_path(path):
    m = re.search(
        r"gene_likelihood=([^,]+),lr=([^,]+),n_hidden=([^,]+),n_latent=([^,]+),n_layers=([^_]+)",
        path or ""
    )
    if not m:
        return {}
    return {
        "gene_likelihood": m.group(1),
        "lr": float(m.group(2)),
        "n_hidden": int(m.group(3)),
        "n_latent": int(m.group(4)),
        "n_layers": int(m.group(5)),
    }

records = []
for r in getattr(results, "results", []):
    loss = (getattr(r, "metrics", {}) or {}).get("validation_loss", math.inf)
    cfg = getattr(r, "config", None) or {}
    if not cfg:
        cfg = extract_from_path(getattr(r, "path", ""))

    records.append({
        "validation_loss": loss,
        "config": cfg,
        "path": getattr(r, "path", None),
        "result_obj": r,
    })

records = [x for x in records if math.isfinite(x["validation_loss"])]
records.sort(key=lambda x: x["validation_loss"])

best = records[0]
print("best validation_loss:", best["validation_loss"])
print("best config:", best["config"])
print("best path:", best["path"])

In [ ]:
# after tuning
top_param = {'n_hidden': 192, 
'n_latent': 30, 
'n_layers': 1, 
'gene_likelihood': 'nb', 
'lr': 0.009785959042550195}

In [ ]:
scvi.model.SCVI.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                              categorical_covariate_keys=['study_id'], 
                              continuous_covariate_keys=['pct_counts_MT', 'total_counts'])
model = scvi.model.SCVI(adata, n_hidden = top_param['n_hidden'], 
                        n_latent = top_param['n_latent'], 
                        n_layers = top_param['n_layers'], 
                        gene_likelihood = top_param['gene_likelihood'])
kwargs = {'lr': top_param['lr']}
model.train(max_epochs = 200, early_stopping = True, plan_kwargs = kwargs)

In [ ]:
y = model.history['reconstruction_loss_validation']['reconstruction_loss_validation'].min()
plt.plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
plt.plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')

plt.axhline(y, c = 'k')

plt.legend()
plt.show()

In [ ]:
model.save(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_scvi_integration_model_dbl'))


In [ ]:
# ensure formatting before write
def fix_stringarray(df):
    """Convert StringDtype index and columns to object dtype."""
    if isinstance(df.index.dtype, pd.StringDtype):
        df.index = df.index.astype(str)
    for col in df.columns:
        if isinstance(df[col].dtype, pd.StringDtype):
            df[col] = df[col].astype(str)
    return df

# Fix obs and var
adata.obs = fix_stringarray(adata.obs)
adata.var = fix_stringarray(adata.var)

# Fix DataFrames in obsm
for key in list(adata.obsm.keys()):
    if isinstance(adata.obsm[key], pd.DataFrame):
        adata.obsm[key] = fix_stringarray(adata.obsm[key])

# Fix DataFrames in varm
for key in list(adata.varm.keys()):
    if isinstance(adata.varm[key], pd.DataFrame):
        adata.varm[key] = fix_stringarray(adata.varm[key])

adata.write_h5ad(str(PBMC_INTERMEDIATE / 'temp_pbmc_myeloid_platelet_dbl.h5ad'))


In [ ]:
adata = sc.read_h5ad(str(PBMC_INTERMEDIATE / 'temp_pbmc_myeloid_platelet_dbl.h5ad'))
model = scvi.model.SCVI.load(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_scvi_integration_model_dbl/'), adata)


In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation()
scvi_norm_count = model.get_normalized_expression(library_size = 1e4)
adata.layers['scvi_normalized'] = scvi_norm_count

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 123)

In [ ]:
sc.tl.leiden(adata, resolution = 2, key_added = 'overcluster')

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.set_figure_params(dpi=200)
sc.pl.umap(adata, color = 'lane', size = 2, legend_fontsize = 6)

In [ ]:
sc.set_figure_params(dpi=500)
sc.pl.umap(adata, color = 'study_id', size = 1, legend_fontsize = 6)

In [ ]:
sc.set_figure_params(dpi=200)
sc.pl.umap(adata, color = 'study_day', size = 2, legend_fontsize = 6)

In [ ]:
sc.set_figure_params(dpi=200)
sc.pl.umap(adata, color = 'souporcell_status', size = 2, legend_fontsize = 6)
#sc.pl.umap(adata, color = 'solo_prediction', size = 1, legend_fontsize = 'medium', palette = ['red','green','blue'])

In [ ]:
sc.set_figure_params(dpi=200)

sc.pl.umap(adata, color = ['CD14','FCGR3A','LYZ','PLD4','ITM2C','S100A9','NEAT1','LILRA4','CLEC9A','HLA-DQA1','FCER1A','IRF4'], size = 4, frameon = False, layer = 'scvi_normalized')

In [ ]:
sc.set_figure_params(dpi=100)
plt.rcParams['figure.figsize'] = [7,7]
sc.pl.umap(adata, color = ['overcluster'], legend_loc = 'on data', size = 3, layer = 'scvi_normalized', legend_fontsize = 'x-small')

In [ ]:
sc.set_figure_params(dpi=100)
sc.pl.dotplot(adata, ['CD3E','CD19','NCAM1','CD14','FCGR3A','LYZ','PLD4','ITM2C','S100A9','NEAT1','LILRA4',
                      'CLEC9A','IDO1','HLA-DQA1','FCER1A','CLEC10A','IRF4','MKI67',
                      'GNG11','PPBP','PF4','CAVIN2','TUBB1'], groupby = 'overcluster', swap_axes = True,
              use_raw = True, standard_scale = 'var', dendrogram = False)

In [ ]:
from pandas.errors import PerformanceWarning

# avoid plethora of message output
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=PerformanceWarning,
        module=r"scanpy\.tools\._rank_genes_groups"
    )
    sc.tl.rank_genes_groups(adata, groupby="overcluster", method="wilcoxon")

In [ ]:
# Convert 'overcluster' column to integer for processing
overcluster_int = adata.obs['overcluster'].astype(int)

# Get unique clusters sorted in ascending order
unique_clusters = sorted(overcluster_int.unique())

# Initialize lists to store the calculated frequencies
singlet_freqs = []
doublet_freqs = []
souporcell_doublet_freqs = []
doublet_sum_freqs = []

# Calculate frequencies for each cluster
for cluster in unique_clusters:
    cluster_data = adata.obs[overcluster_int == cluster]
    total_cells = len(cluster_data)

    singlet_freq = (cluster_data['souporcell_status'] == 'singlet').sum() / total_cells * 100
    souporcell_doublet_freq = (cluster_data['souporcell_status'] == 'doublet').sum() / total_cells * 100
    doublet_freq = 0.0
    doublet_sum_freq = souporcell_doublet_freq

    singlet_freqs.append(singlet_freq)
    doublet_freqs.append(doublet_freq)
    souporcell_doublet_freqs.append(souporcell_doublet_freq)
    doublet_sum_freqs.append(doublet_sum_freq)

# Calculate overall frequencies
total_cells = len(adata.obs)
overall_singlet_freq = (adata.obs['souporcell_status'] == 'singlet').sum() / total_cells * 100
overall_souporcell_doublet_freq = (adata.obs['souporcell_status'] == 'doublet').sum() / total_cells * 100
overall_doublet_freq = 0.0
overall_doublet_sum_freq = overall_souporcell_doublet_freq

# Create the DataFrame for clusters
report_df = pd.DataFrame({
    'overcluster': [str(cluster) for cluster in unique_clusters],
    'singlet': singlet_freqs,
    'doublet': doublet_freqs,
    'souporcell_doublet': souporcell_doublet_freqs,
    'doublet_sum': doublet_sum_freqs,
})

# Add overall frequencies to the DataFrame
overall_df = pd.DataFrame({
    'overcluster': ['overall'],
    'singlet': [overall_singlet_freq],
    'doublet': [overall_doublet_freq],
    'souporcell_doublet': [overall_souporcell_doublet_freq],
    'doublet_sum': [overall_doublet_sum_freq],
})

# Use pd.concat to append the overall frequencies to the report DataFrame
final_report_df = pd.concat([report_df, overall_df], ignore_index=True)

print(final_report_df)


In [ ]:
singlet_rate_map = dict(zip(final_report_df['overcluster'], final_report_df['singlet']))
doublet_rate_map = dict(zip(final_report_df['overcluster'], final_report_df['doublet']))
souporcell_doublet_rate_map = dict(zip(final_report_df['overcluster'], final_report_df['souporcell_doublet']))
doublet_sum_rate_map = dict(zip(final_report_df['overcluster'], final_report_df['doublet_sum']))


In [ ]:
adata.obs['singlet_frequency'] = adata.obs['overcluster'].map(singlet_rate_map).astype(float)
adata.obs['doublet_frequency'] = adata.obs['overcluster'].map(doublet_rate_map).astype(float)
adata.obs['souporcell_doublet_frequency'] = adata.obs['overcluster'].map(souporcell_doublet_rate_map).astype(float)
adata.obs['doublet_sum_frequency'] = adata.obs['overcluster'].map(doublet_sum_rate_map).astype(float)
adata.obs['solo_prediction'] = np.where(
    adata.obs['souporcell_status'].astype(str).eq('doublet'),
    'souporcell_doublet',
    'singlet'
)


In [ ]:
sc.set_figure_params(dpi=100)
plt.rcParams['figure.figsize'] = [5,5]
sc.pl.umap(adata, color = 'singlet_frequency', size = 3)

In [ ]:
myeloid_platelet_map = {
    '0': 'nMono', #
    '1': 'cMono', #
    '2': 'Doublet', #
    '3': 'cMono', #
    '4': 'cMono', #
    '5': 'Doublet', #
    '6': 'cMono', #
    '7': 'cMono', #
    '8': 'cMono', #
    '9': 'cMono', #
    '10': 'cMono', #
    '11': 'nMono', #
    '12': 'Doublet', # NK
    '13': 'MPA', #
    '14': 'cDC2', #
    '15': 'Platelet', #
    '16': 'Doublet', #
    '17': 'Doublet', # B
    '18': 'cMono', #
    '19': 'pDC', #
    '20': 'Doublet', #
    '21': 'other', #
    '22': 'cDC1', # 
}

In [ ]:
adata.obs['ann_types'] = adata.obs['overcluster'].map(myeloid_platelet_map)
plt.rcParams['figure.figsize'] = [10, 8]
sc.pl.umap(adata, color = ['ann_types'], legend_loc = 'on data', size = 5, legend_fontsize = 15)

In [ ]:
adata.write_h5ad(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_int_dbl.h5ad'))


In [ ]:
np.savetxt(str(REPO_ROOT / 'intermediate' / 'pbmc_myeloid_platelet_int_dbl_umap_coordinates_to_r.csv'), adata.obsm['X_umap'], delimiter = ',')
adata.obs[['barcode_2', 'ann_types', 'souporcell_status', 'study_id', 'study_day',
           'solo_prediction', 'doublet_frequency', 'souporcell_doublet_frequency',
           'doublet_sum_frequency', 'singlet_frequency']].to_csv(
               str(REPO_ROOT / 'intermediate' / 'pbmc_myeloid_platelet_int_dbl_obs_to_r.csv'),
               index=False
           )


In [ ]:
assert np.all(adata.layers['counts'].data == adata.layers['counts'].data.astype(np.int32))
mmwrite(target=str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_dbl_counts.mtx'), a=adata.layers['counts'])
adata.obs.to_csv(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_dbl_obs.csv'))
adata.var.to_csv(str(PBMC_INTERMEDIATE / 'pbmc_myeloid_platelet_dbl_var.csv'))
